# 07. Uncertainty Quantification and Cross-Validation Audit

This notebook responds to the professor's feedback on uncertainty quantification and cross-validation design for the IDS705 final report.

It uses the locked outputs from `06_final_report_model_lock.ipynb` whenever possible, so the analysis answers a narrow question: given the already selected model predictions on the 2018 and 2022 holdout tournaments, how much uncertainty surrounds the reported differences?

Scope:

1. Table 1 overall holdout uncertainty for accuracy and macro-F1.
2. Stage-wise uncertainty for group-stage and knockout-stage comparisons.
3. Paired bootstrap differences between team-history and player-enriched models.
4. Aggregate upset diagnostics, replacing anecdote-only evidence.
5. Confederation-level reliability intervals.
6. Cross-validation timing audit for the leave-one-tournament-out design.

The goal is exploratory: produce evidence tables and figures first, then decide how strongly the final report should word each claim.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3

import numpy as np
import pandas as pd
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
import matplotlib
if "get_ipython" not in globals():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

NOTEBOOK_DIR = Path.cwd()
LOCK_DIR = NOTEBOOK_DIR / "data" / "final_report_lock"
OUT_DIR = NOTEBOOK_DIR / "data" / "uncertainty_audit"
FIG_DIR = NOTEBOOK_DIR / "docs" / "figures" / "uncertainty_audit"
DB_PATH = NOTEBOOK_DIR / "../../../../data/raw/worldcup/data-sqlite/worldcup.db"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = [0, 1, 2]
CLASS_NAMES = {0: "fav_loses", 1: "draw", 2: "fav_wins"}
MODEL_KEYS = ["baseline", "team_history", "player_enriched"]
MODEL_NAMES = {
    "baseline": "Baseline",
    "team_history": "Team-history",
    "player_enriched": "Player-enriched",
}
PAIRWISE_FOCUS = ("player_enriched", "team_history")
BOOTSTRAP_B = 5000
SEED = 705

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("LOCK_DIR exists:", LOCK_DIR.exists())
print("OUT_DIR:", OUT_DIR)

## 1. Load Locked Predictions

The holdout file has one row per 2018/2022 World Cup match, plus each model's prediction and class probabilities. These rows are the unit resampled in the bootstrap.

In [ ]:
holdout = pd.read_parquet(LOCK_DIR / "holdout_predictions.parquet")
table1 = pd.read_csv(LOCK_DIR / "table1_summary.csv")
stage_compact = pd.read_csv(LOCK_DIR / "stage_summary_compact.csv")
specs = json.loads((LOCK_DIR / "final_model_specs.json").read_text())

holdout["stage"] = np.where(holdout["knockout_stage"].astype(bool), "knockout_stage", "group_stage")
holdout["actual"] = holdout["y"].map(CLASS_NAMES)

print("Holdout shape:", holdout.shape)
display(holdout.head())
display(holdout.groupby(["year", "stage"]).size().to_frame("n_matches"))
display(holdout["actual"].value_counts().rename("n").to_frame())

## 2. Baseline Context

Before discussing model differences, we need simple reference points:

- **Majority-class baseline:** always predict favorite wins.
- **Empirical random baseline:** randomly guess according to the holdout class distribution.
- **Uniform random baseline:** randomly guess each of the three outcomes equally.

These contextualize whether the reported 65% accuracy is high, moderate, or mostly reflecting the class imbalance.

In [ ]:
def metric_row(y_true, y_pred, model_key, stage="overall"):
    return {
        "model_key": model_key,
        "stage": stage,
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


context_rows = []
y = holdout["y"].to_numpy()
majority_pred = np.full_like(y, fill_value=2)
majority = metric_row(y, majority_pred, "majority_favorite_wins")
context_rows.append({
    "model_key": majority["model_key"],
    "stage": majority["stage"],
    "n": majority["n"],
    "accuracy_mean": majority["accuracy"],
    "accuracy_ci_low": majority["accuracy"],
    "accuracy_ci_high": majority["accuracy"],
    "macro_f1_mean": majority["macro_f1"],
    "macro_f1_ci_low": majority["macro_f1"],
    "macro_f1_ci_high": majority["macro_f1"],
})

class_probs = holdout["y"].value_counts(normalize=True).sort_index().reindex(CLASS_LABELS, fill_value=0).to_numpy()
rng = np.random.default_rng(SEED)
for name, probs in [
    ("random_empirical_distribution", class_probs),
    ("random_uniform_three_class", np.repeat(1 / 3, 3)),
]:
    accs, f1s = [], []
    for _ in range(BOOTSTRAP_B):
        pred = rng.choice(CLASS_LABELS, size=len(y), replace=True, p=probs)
        accs.append(accuracy_score(y, pred))
        f1s.append(f1_score(y, pred, average="macro", zero_division=0))
    context_rows.append({
        "model_key": name,
        "stage": "overall",
        "n": int(len(y)),
        "accuracy_mean": float(np.mean(accs)),
        "accuracy_ci_low": float(np.percentile(accs, 2.5)),
        "accuracy_ci_high": float(np.percentile(accs, 97.5)),
        "macro_f1_mean": float(np.mean(f1s)),
        "macro_f1_ci_low": float(np.percentile(f1s, 2.5)),
        "macro_f1_ci_high": float(np.percentile(f1s, 97.5)),
    })

context_df = pd.DataFrame(context_rows)
context_df.to_csv(OUT_DIR / "context_baselines.csv", index=False)
display(context_df)

## 3. Bootstrap Helpers

We use a paired bootstrap: each bootstrap sample is a set of holdout row indices, and all models are evaluated on that same sample. This preserves model-to-model pairing and lets us quantify uncertainty around both individual metrics and model differences.

In [ ]:
def evaluate_model(df, model_key):
    y_true = df["y"].to_numpy()
    y_pred = df[f"{model_key}_pred"].to_numpy()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def summarize_samples(values):
    values = np.asarray(values, dtype=float)
    return {
        "mean": float(np.mean(values)),
        "sd": float(np.std(values, ddof=1)),
        "minus_2sd": float(np.mean(values) - 2 * np.std(values, ddof=1)),
        "plus_2sd": float(np.mean(values) + 2 * np.std(values, ddof=1)),
        "ci_low": float(np.percentile(values, 2.5)),
        "ci_high": float(np.percentile(values, 97.5)),
    }


def bootstrap_metrics(df, model_keys=MODEL_KEYS, stages=("overall",), b=BOOTSTRAP_B, seed=SEED):
    rng = np.random.default_rng(seed)
    records = []
    diff_records = []

    stage_masks = {"overall": np.ones(len(df), dtype=bool)}
    stage_masks.update({
        "group_stage": df["group_stage"].astype(bool).to_numpy(),
        "knockout_stage": df["knockout_stage"].astype(bool).to_numpy(),
    })

    for stage in stages:
        base_idx = np.flatnonzero(stage_masks[stage])
        n = len(base_idx)
        metric_samples = {
            (model_key, metric): []
            for model_key in model_keys
            for metric in ["accuracy", "macro_f1"]
        }
        diff_samples = {metric: [] for metric in ["accuracy", "macro_f1"]}

        for _ in range(b):
            sample_idx = rng.choice(base_idx, size=n, replace=True)
            sample = df.iloc[sample_idx]

            current = {}
            for model_key in model_keys:
                current[model_key] = evaluate_model(sample, model_key)
                for metric, value in current[model_key].items():
                    metric_samples[(model_key, metric)].append(value)

            hi, lo = PAIRWISE_FOCUS
            for metric in diff_samples:
                diff_samples[metric].append(current[hi][metric] - current[lo][metric])

        for model_key in model_keys:
            point = evaluate_model(df.iloc[base_idx], model_key)
            for metric in ["accuracy", "macro_f1"]:
                stats = summarize_samples(metric_samples[(model_key, metric)])
                records.append({
                    "stage": stage,
                    "model_key": model_key,
                    "model_name": MODEL_NAMES[model_key],
                    "metric": metric,
                    "n": n,
                    "point_estimate": point[metric],
                    **stats,
                })

        hi, lo = PAIRWISE_FOCUS
        for metric in ["accuracy", "macro_f1"]:
            point_hi = evaluate_model(df.iloc[base_idx], hi)[metric]
            point_lo = evaluate_model(df.iloc[base_idx], lo)[metric]
            stats = summarize_samples(diff_samples[metric])
            diff_records.append({
                "stage": stage,
                "comparison": f"{MODEL_NAMES[hi]} - {MODEL_NAMES[lo]}",
                "metric": metric,
                "n": n,
                "point_difference": point_hi - point_lo,
                **stats,
            })

    return pd.DataFrame(records), pd.DataFrame(diff_records)

## 4. Table 1 Uncertainty

This section quantifies uncertainty for the overall 2018/2022 holdout metrics reported in Table 1.

In [ ]:
overall_uncertainty, overall_differences = bootstrap_metrics(
    holdout,
    model_keys=MODEL_KEYS,
    stages=("overall",),
)

overall_uncertainty.to_csv(OUT_DIR / "table1_overall_uncertainty.csv", index=False)
overall_differences.to_csv(OUT_DIR / "table1_pairwise_differences.csv", index=False)

display(
    overall_uncertainty
    .pivot_table(index=["model_name", "n"], columns="metric", values=["point_estimate", "ci_low", "ci_high", "minus_2sd", "plus_2sd"])
    .round(3)
)
display(overall_differences.round(3))

## 5. Stage-Wise Uncertainty

This targets the Figure 3 / Table 2 comments. The key issue is whether the knockout-stage advantage is larger than bootstrap uncertainty, especially because the knockout sample has only 32 matches.

In [ ]:
stage_uncertainty, stage_differences = bootstrap_metrics(
    holdout,
    model_keys=["team_history", "player_enriched"],
    stages=("group_stage", "knockout_stage", "overall"),
)

stage_uncertainty.to_csv(OUT_DIR / "stage_uncertainty.csv", index=False)
stage_differences.to_csv(OUT_DIR / "stage_pairwise_differences.csv", index=False)

display(stage_uncertainty.round(3))
display(stage_differences.round(3))

In [ ]:
def plot_interval(df, metric, title, path):
    plot_df = df[df["metric"] == metric].copy()
    order = ["group_stage", "knockout_stage", "overall"]
    model_order = ["team_history", "player_enriched"] if "team_history" in set(plot_df["model_key"]) else MODEL_KEYS
    colors = {"baseline": "#6B7280", "team_history": "#2563EB", "player_enriched": "#DC2626"}

    fig, ax = plt.subplots(figsize=(8, 4.8))
    x_base = np.arange(len(order))
    width = 0.18
    offsets = np.linspace(-width, width, len(model_order))

    for offset, model_key in zip(offsets, model_order):
        sub = plot_df[plot_df["model_key"] == model_key].set_index("stage").reindex(order).dropna(subset=["point_estimate"])
        x = np.array([order.index(stage) for stage in sub.index]) + offset
        y = sub["point_estimate"].to_numpy()
        yerr = np.vstack([
            y - sub["ci_low"].to_numpy(),
            sub["ci_high"].to_numpy() - y,
        ])
        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt="o",
            capsize=4,
            linewidth=1.5,
            markersize=6,
            color=colors[model_key],
            label=MODEL_NAMES[model_key],
        )

    ax.set_xticks(x_base)
    ax.set_xticklabels([s.replace("_", " ").title() for s in order])
    ax.set_ylim(0, 1)
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()


plot_interval(
    stage_uncertainty,
    "accuracy",
    "Stage-wise holdout accuracy with paired bootstrap 95% intervals",
    FIG_DIR / "stage_accuracy_intervals.png",
)
plot_interval(
    stage_uncertainty,
    "macro_f1",
    "Stage-wise holdout macro-F1 with paired bootstrap 95% intervals",
    FIG_DIR / "stage_macro_f1_intervals.png",
)

## 6. Aggregate Upset Diagnostics

The report currently uses a few illustrative upset examples. This section converts that into aggregate evidence:

- actual upset matches are rows where the favorite lost (`y == fav_loses`);
- compare whether the player-enriched model lowers favorite-win probability relative to team-history;
- count whether either model correctly classifies the upset.

In [ ]:
upsets = holdout[holdout["y"] == 0].copy()
upsets["delta_favorite_win_prob_player_minus_team"] = (
    upsets["player_enriched_prob_fav_wins"] - upsets["team_history_prob_fav_wins"]
)
upsets["player_lowered_favorite_win_prob"] = upsets["delta_favorite_win_prob_player_minus_team"] < 0
upsets["team_history_correct"] = upsets["team_history_pred"] == upsets["y"]
upsets["player_enriched_correct"] = upsets["player_enriched_pred"] == upsets["y"]
upsets["player_corrected_team_error"] = upsets["player_enriched_correct"] & ~upsets["team_history_correct"]
upsets["player_missed_team_correct"] = ~upsets["player_enriched_correct"] & upsets["team_history_correct"]

upset_summary = pd.DataFrame([{
    "n_actual_upsets": len(upsets),
    "n_player_lowered_favorite_win_prob": int(upsets["player_lowered_favorite_win_prob"].sum()),
    "share_player_lowered_favorite_win_prob": float(upsets["player_lowered_favorite_win_prob"].mean()),
    "mean_delta_favorite_win_prob_player_minus_team": float(upsets["delta_favorite_win_prob_player_minus_team"].mean()),
    "median_delta_favorite_win_prob_player_minus_team": float(upsets["delta_favorite_win_prob_player_minus_team"].median()),
    "team_history_correct_upsets": int(upsets["team_history_correct"].sum()),
    "player_enriched_correct_upsets": int(upsets["player_enriched_correct"].sum()),
    "player_corrected_team_error": int(upsets["player_corrected_team_error"].sum()),
    "player_missed_team_correct": int(upsets["player_missed_team_correct"].sum()),
}])

upset_summary.to_csv(OUT_DIR / "upset_aggregate_summary.csv", index=False)
upsets.to_csv(OUT_DIR / "upset_match_level_diagnostics.csv", index=False)

display(upset_summary)
display(
    upsets[[
        "match_id", "year", "stage",
        "team_history_prob_fav_wins", "player_enriched_prob_fav_wins",
        "delta_favorite_win_prob_player_minus_team",
        "team_history_correct", "player_enriched_correct",
    ]].sort_values("delta_favorite_win_prob_player_minus_team").head(10)
)

## 7. Confederation Reliability Intervals

This addresses the small-n regional reliability comment. Accuracy by confederation should be shown with sample size and uncertainty. The interval is a bootstrap interval within each confederation group; for tiny groups, it will often be very wide.

In [ ]:
con = sqlite3.connect(DB_PATH)
teams = pd.read_sql_query("SELECT team_id, team_name, confederation_id FROM teams", con)
confederations = pd.read_sql_query(
    "SELECT confederation_id, confederation_code, confederation_name FROM confederations",
    con,
)
con.close()

conf_df = holdout.merge(
    teams.rename(columns={
        "team_id": "fav_team_id",
        "team_name": "fav_team_name",
        "confederation_id": "fav_confederation",
    }),
    on="fav_team_id",
    how="left",
)

def bootstrap_group_accuracy(df, model_key, b=BOOTSTRAP_B, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true = df["y"].to_numpy()
    y_pred = df[f"{model_key}_pred"].to_numpy()
    n = len(df)
    point = accuracy_score(y_true, y_pred)
    vals = []
    for _ in range(b):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        vals.append(accuracy_score(y_true[idx], y_pred[idx]))
    stats = summarize_samples(vals)
    return {"n": n, "point_estimate": point, **stats}


conf_df = conf_df.merge(
    confederations.rename(columns={"confederation_id": "fav_confederation"}),
    on="fav_confederation",
    how="left",
)
conf_df["confederation_label"] = conf_df["confederation_code"].fillna(conf_df["fav_confederation"])

conf_rows = []
for conf_idx, (confederation, sub) in enumerate(conf_df.groupby("fav_confederation", dropna=False)):
    for model_idx, model_key in enumerate(["team_history", "player_enriched"]):
        stats = bootstrap_group_accuracy(sub, model_key, seed=SEED + conf_idx * 100 + model_idx)
        conf_rows.append({
            "fav_confederation": confederation,
            "confederation_code": sub["confederation_code"].iloc[0],
            "confederation_name": sub["confederation_name"].iloc[0],
            "model_key": model_key,
            "model_name": MODEL_NAMES[model_key],
            **stats,
        })

conf_uncertainty = pd.DataFrame(conf_rows).sort_values(["fav_confederation", "model_key"])
conf_uncertainty.to_csv(OUT_DIR / "confederation_accuracy_uncertainty.csv", index=False)
display(conf_uncertainty.round(3))

In [ ]:
plot_df = conf_uncertainty.copy()
conf_order = (
    plot_df.groupby("fav_confederation")["n"].max()
    .sort_values(ascending=False)
    .index.tolist()
)
colors = {"team_history": "#2563EB", "player_enriched": "#DC2626"}

fig, ax = plt.subplots(figsize=(8, 4.8))
x_base = np.arange(len(conf_order))
offsets = {"team_history": -0.08, "player_enriched": 0.08}

for model_key in ["team_history", "player_enriched"]:
    sub = plot_df[plot_df["model_key"] == model_key].set_index("fav_confederation").reindex(conf_order)
    x = x_base + offsets[model_key]
    y = sub["point_estimate"].to_numpy()
    yerr = np.vstack([y - sub["ci_low"].to_numpy(), sub["ci_high"].to_numpy() - y])
    ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=4, color=colors[model_key], label=MODEL_NAMES[model_key])

labels = []
for conf in conf_order:
    conf_sub = plot_df[plot_df["fav_confederation"] == conf]
    code = conf_sub["confederation_code"].iloc[0]
    labels.append(f"{code}\n(n={int(conf_sub['n'].max())})")
ax.set_xticks(x_base)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by favorite-team confederation with bootstrap 95% intervals")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "confederation_accuracy_intervals.png", dpi=200)
plt.show()

## 8. Cross-Validation Timing Audit

The locked report uses leave-one-tournament-out cross-validation on the pre-holdout training era. This is useful for estimating tournament-to-tournament stability, but it is not a strictly prospective forecast simulation: when the held-out fold is an earlier World Cup, the training folds include later tournaments.

This matters because the stakeholder story is prospective prediction. We should decide whether to:

1. keep LOTO CV but explicitly label it as a stability check, while relying on 2018/2022 holdout as the true prospective test; or
2. add a forward-chaining CV sensitivity check where each tournament is predicted only from earlier tournaments.

The table below identifies which LOTO folds train on future tournaments relative to the fold being predicted.

In [ ]:
cv_rows = []
for model_key, spec in specs.items():
    train_min_year = int(spec["train_min_year"])
    cv_years = [year for year in sorted(holdout["year"].unique()) if year < min(holdout["year"])]
    # The locked pre-holdout World Cup years are inferred from each model's training window.
    if train_min_year == 1998:
        cv_years = [1998, 2002, 2006, 2010, 2014]
    elif train_min_year == 2006:
        cv_years = [2006, 2010, 2014]
    else:
        raise ValueError(f"Unexpected train_min_year for {model_key}: {train_min_year}")

    for heldout_year in cv_years:
        train_years = [year for year in cv_years if year != heldout_year]
        future_train_years = [year for year in train_years if year > heldout_year]
        past_train_years = [year for year in train_years if year < heldout_year]
        cv_rows.append({
            "model_key": model_key,
            "model_name": specs[model_key]["display_name"],
            "heldout_year": heldout_year,
            "loto_train_years": ", ".join(map(str, train_years)),
            "uses_future_tournaments": len(future_train_years) > 0,
            "future_train_years": ", ".join(map(str, future_train_years)),
            "past_train_years": ", ".join(map(str, past_train_years)),
            "forward_chaining_possible": len(past_train_years) > 0,
        })

cv_timing_audit = pd.DataFrame(cv_rows)
cv_timing_audit.to_csv(OUT_DIR / "cv_timing_audit.csv", index=False)
display(cv_timing_audit)
display(
    cv_timing_audit.groupby("model_name")["uses_future_tournaments"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "folds_using_future_tournaments", "count": "total_loto_folds"})
)

## 9. Optional Next Step: Forward-Chaining Sensitivity

If we decide the CV critique needs empirical remediation rather than a written clarification, run a forward-chaining sensitivity check in a follow-up notebook or appended section.

Suggested design:

- For each candidate model, keep the same feature set and model family.
- For each tournament year after the first available training tournament, train only on strictly earlier tournaments.
- Report fold-level accuracy and macro-F1.
- Use this as a sensitivity analysis, not as a replacement for the locked 2018/2022 holdout.

Important wording if we keep LOTO:

> Leave-one-tournament-out cross-validation was used as a pre-holdout stability check, not as a simulated prospective forecast. The prospective evaluation is the final holdout on the 2018 and 2022 tournaments, where no future tournament information is used for model fitting.

## 10. Reading Guide for Report Revision

After running this notebook, use:

- `data/uncertainty_audit/table1_overall_uncertainty.csv`
- `data/uncertainty_audit/table1_pairwise_differences.csv`
- `data/uncertainty_audit/stage_uncertainty.csv`
- `data/uncertainty_audit/stage_pairwise_differences.csv`
- `data/uncertainty_audit/upset_aggregate_summary.csv`
- `data/uncertainty_audit/confederation_accuracy_uncertainty.csv`
- `data/uncertainty_audit/cv_timing_audit.csv`

Decision rule for narrative strength:

- If a difference interval crosses zero, describe the model as having a higher point estimate but not decisive evidence.
- If intervals are wide because `n` is small, frame the result as a diagnostic pattern rather than a generalizable conclusion.
- If the upset aggregate is weak, keep individual examples but label them as illustrative cases, not broad evidence.